In [ ]:
%cd ..

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import pandas as pd
import zarr
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from common.sample_db import SampleDB

db = SampleDB()

### distributiuon of losses from score

In [ ]:
SOBOL_RUN = "004"

SCORE_MODEL_IS_RUNS = [
    ('231_keras/i0', "230"),   # this is a special case, gives baseline performance before IS training on hard egs
    ('231_keras/i1', "210"),
    ('231_keras/i2', "230")
]

BETA_STFT = 0.01

In [ ]:
dfs = []

for score_model, is_run in SCORE_MODEL_IS_RUNS:

    losses = db.losses_for(run=SOBOL_RUN, model=score_model)
    df = pd.DataFrame(losses)
    df['run'] = SOBOL_RUN
    df['score_model'] = score_model
    df['src'] = 'sobol'
    dfs.append(df)

    if is_run is not None:
        losses = db.losses_for(run=is_run, model=score_model)
        df = pd.DataFrame(losses)
        df['run'] = is_run
        df['score_model'] = score_model
        df['src'] = 'hard_mined'
        dfs.append(df)

losses_df = pd.concat(dfs)
losses_df.head()
#losses_1_df['src'] = 'sobol'
#losses_1_df.describe()

In [ ]:
losses_df.score_model.unique()

In [ ]:
src_values = ['sobol', 'hard_mined']
cols = ['huber', 'stft']
score_models = sorted(losses_df['score_model'].unique())

shared_xlim = {
    col: (losses_df[col].min(), losses_df[col].max())
    for col in cols
}

fig, axes = plt.subplots(len(src_values), len(cols), figsize=(12, 4 * len(src_values)), squeeze=False)

for row, src in enumerate(src_values):
    subset = losses_df[losses_df['src'] == src]
    for col_idx, col in enumerate(cols):
        ax = axes[row, col_idx]
        sns.histplot(
            data=subset,
            x=col,
            hue='score_model',
            hue_order=score_models,
            kde=True,
            stat='density',
            common_norm=False,
            alpha=0.35,
            element='step',
            ax=ax,
        )
        ax.set_xlim(shared_xlim[col])

# Column labels (top)
for col_idx, col in enumerate(cols):
    x = (col_idx + 0.5) / len(cols)
    fig.text(x, 0.995, col, ha='center', va='top', fontsize=12, fontweight='bold')

# Row labels (left, vertical)
for row, src in enumerate(src_values):
    y = 1 - ((row + 0.5) / len(src_values))
    fig.text(0.028, y, src, ha='center', va='center', rotation=90, fontsize=12, fontweight='bold')

plt.tight_layout(rect=(0.07, 0.03, 1.0, 0.96))
plt.show()